## Dados simulados: vendas de ingressos na noite de Halloween

A base do RA1 é estática: cada linha é um filme, e nenhum registro novo chega com o tempo. Para aplicar streaming, criamos registros **artificiais** de venda de ingressos em um cinema na noite de 31/10/2022. Os horários e os valores foram inventados, mas os filmes existem na base limpa do RA1 e estavam em cartaz naquele mês.

Cada venda tem três campos:

| Campo | Significado | Exemplo |
|---|---|---|
| `timestamp` | quando a venda aconteceu | 2022-10-31 20:04:00 |
| `filme` | título do filme na base do RA1 | Terrifier 2 |
| `valor` | valor da venda em reais | 30 |

O ingresso custa R$ 30 (inteira) ou R$ 15 (meia), então cada valor corresponde a uma compra de um ou mais ingressos.

In [1]:
import pandas as pd

URL_LIMPO = "https://raw.githubusercontent.com/carlaferraz/HorrorMoviesData/refs/heads/main/horror_movies_limpo.csv"
filmes_base = pd.read_csv(URL_LIMPO, usecols=["id", "title", "release_date", "vote_average", "vote_count"])

filmes_em_cartaz = ["Smile", "Halloween Ends", "Terrifier 2", "Barbarian"]
ids_em_cartaz = [882598, 616820, 663712, 913290]

filmes_base[filmes_base["id"].isin(ids_em_cartaz)]

,id,title,release_date,vote_count,vote_average
2,882598,Smile,2022-09-23,114,6.8
28,616820,Halloween Ends,2022-10-12,2,10.0
100,663712,Terrifier 2,2022-10-06,1,8.0
114,913290,Barbarian,2022-09-08,61,7.2


**Leitura:** os quatro filmes estão na base limpa e foram lançados entre setembro e outubro de 2022, então faz sentido que estivessem em cartaz no Halloween. Usamos os IDs na conferência com a base porque existe mais de um filme chamado *Smile* (Q1 mostrou que título não identifica o filme).

In [2]:
vendas_etapa1 = [
    ("2022-10-31 20:01:00", "Smile", 60),
    ("2022-10-31 20:03:00", "Halloween Ends", 45),
    ("2022-10-31 20:04:00", "Terrifier 2", 30),
]

vendas_etapa2 = [
    ("2022-10-31 20:06:00", "Barbarian", 90),
    ("2022-10-31 20:07:00", "Smile", 15),
]

vendas_etapa3 = [
    ("2022-10-31 20:08:00", "Halloween Ends", 60),
    ("2022-10-31 20:09:00", "Terrifier 2", 45),
    ("2022-10-31 20:10:00", "Smile", 30),
]

todas_vendas = vendas_etapa1 + vendas_etapa2 + vendas_etapa3
assert all(filme in filmes_em_cartaz for _, filme, _ in todas_vendas)

pd.DataFrame(todas_vendas, columns=["timestamp", "filme", "valor"])

,timestamp,filme,valor
0,2022-10-31 20:01:00,Smile,60
1,2022-10-31 20:03:00,Halloween Ends,45
2,2022-10-31 20:04:00,Terrifier 2,30
3,2022-10-31 20:06:00,Barbarian,90
4,2022-10-31 20:07:00,Smile,15
5,2022-10-31 20:08:00,Halloween Ends,60
6,2022-10-31 20:09:00,Terrifier 2,45
7,2022-10-31 20:10:00,Smile,30


**Leitura:** são 8 vendas entre 20:01 e 20:10, somando R$ 375. Elas chegam em três etapas, como se o sistema do cinema enviasse as vendas aos poucos:

| Etapa | Vendas que chegam | Receita da etapa (R$) |
|---|---|---:|
| 1 | 20:01 Smile (60), 20:03 Halloween Ends (45), 20:04 Terrifier 2 (30) | 135 |
| 2 | 20:06 Barbarian (90), 20:07 Smile (15) | 105 |
| 3 | 20:08 Halloween Ends (60), 20:09 Terrifier 2 (45), 20:10 Smile (30) | 135 |

As vendas estão a um ou dois minutos de distância, então várias caem em mais de uma janela, e cada etapa muda janelas que já existiam além de criar novas.

## Janelas temporais

**Tipo:** Sliding Window (janela deslizante).

**Duração:** 5 minutos. **Deslocamento:** 2 minutos. No Spark: `window("timestamp", "5 minutes", "2 minutes")`.

**Agregação:** soma de `valor` em cada janela, que dá a receita do cinema naquele intervalo. O filme não entra no agrupamento: a pergunta é quanto o cinema arrecadou, não quanto cada filme arrecadou.

**Como as janelas são formadas:** uma nova janela começa a cada 2 minutos (20:00, 20:02, 20:04...) e cada uma dura 5 minutos. O início está incluído e o fim está excluído: a janela 20:00–20:05 conta uma venda das 20:04, mas não uma das 20:05. Como o deslocamento é menor que a duração, as janelas se sobrepõem, e uma venda entra em 2 ou 3 janelas. A venda das 20:04, por exemplo, está em 20:00–20:05, 20:02–20:07 e 20:04–20:09.

**Por que essa escolha:**
- Na noite de Halloween as compras se concentram nos minutos antes de cada sessão. Uma janela de 5 minutos pega um desses picos inteiro sem misturar com o próximo.
- O deslocamento de 2 minutos atualiza a receita dos "últimos 5 minutos" com frequência. Com uma Tumbling Window de 5 minutos, o resultado só mudaria de 5 em 5 minutos e um pico que caísse na divisa entre duas janelas ficaria partido ao meio.
- Duração e deslocamento são os mesmos do notebook da aula, o que facilita comparar o resultado.

**Cuidado na leitura:** como a mesma venda aparece em várias janelas, somar as receitas de todas as janelas não dá a receita da noite. A soma das janelas na etapa 3 é R$ 960, mas o cinema vendeu R$ 375.

### Gabarito: resultado esperado em cada etapa

Calculado à mão, antes de rodar o streaming, para conferir a saída do Spark. A tabela é acumulada: cada etapa mostra todas as janelas com as vendas que chegaram até ali. Em negrito, o que mudou em relação à etapa anterior.

**Etapa 1** (4 janelas)

| Janela | Vendas incluídas | Receita (R$) |
|---|---|---:|
| 19:58–20:03 | 20:01 (60) | 60 |
| 20:00–20:05 | 20:01 (60), 20:03 (45), 20:04 (30) | 135 |
| 20:02–20:07 | 20:03 (45), 20:04 (30) | 75 |
| 20:04–20:09 | 20:04 (30) | 30 |

A venda das 20:03 não entra em 19:58–20:03 porque o fim da janela é excluído.

**Etapa 2** (5 janelas)

| Janela | Vendas incluídas | Receita (R$) |
|---|---|---:|
| 19:58–20:03 | 20:01 (60) | 60 |
| 20:00–20:05 | 20:01 (60), 20:03 (45), 20:04 (30) | 135 |
| 20:02–20:07 | 20:03 (45), 20:04 (30), **20:06 (90)** | **165** |
| 20:04–20:09 | 20:04 (30), **20:06 (90), 20:07 (15)** | **135** |
| **20:06–20:11** | **20:06 (90), 20:07 (15)** | **105** |

A venda das 20:06 aumenta duas janelas antigas e abre uma nova.

**Etapa 3** (7 janelas)

| Janela | Vendas incluídas | Receita (R$) |
|---|---|---:|
| 19:58–20:03 | 20:01 (60) | 60 |
| 20:00–20:05 | 20:01 (60), 20:03 (45), 20:04 (30) | 135 |
| 20:02–20:07 | 20:03 (45), 20:04 (30), 20:06 (90) | 165 |
| 20:04–20:09 | 20:04 (30), 20:06 (90), 20:07 (15), **20:08 (60)** | **195** |
| 20:06–20:11 | 20:06 (90), 20:07 (15), **20:08 (60), 20:09 (45), 20:10 (30)** | **240** |
| **20:08–20:13** | **20:08 (60), 20:09 (45), 20:10 (30)** | **135** |
| **20:10–20:15** | **20:10 (30)** | **30** |

A janela 20:06–20:11 é o pico da noite (R$ 240), porque reúne cinco vendas seguidas. As três primeiras janelas não mudam mais, porque nenhuma venda nova cai nelas.

In [3]:
# Verifica se o PySpark está disponível no ambiente.
# No Colab, a instalação é realizada apenas na primeira execução da sessão.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("pyspark") is None:
    print("PySpark não encontrado. Iniciando a instalação...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark"])
else:
    print("PySpark já está instalado.")

PySpark já está instalado.


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import window, sum as soma, to_timestamp

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("HorrorMoviesNight")
    .config("spark.sql.session.timeZone", "America/Sao_Paulo")
    .config("spark.sql.shuffle.partitions", "2")
    # Adicionei para esconder a barra de progresso do Spark, que enfeiava o PDF.
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")
spark.sparkContext.setLogLevel("ERROR")
print("Fuso da sessão:", spark.conf.get("spark.sql.session.timeZone"))


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/19 21:25:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Fuso da sessão: America/Sao_Paulo


# Sliding Window

## Versão estática conferida com o gabarito

Antes de ligar o streaming, a mesma agregação roda em um DataFrame comum com as vendas acumuladas até cada etapa. O gabarito do Baiano vira um dicionário e a comparação é feita com `assert`: se a janela estiver errada, a célula para com erro.

In [5]:
# Substituí a versão estática anterior (só a etapa 1, conferida do zoio) por esta,
# que roda as 3 etapas e compara com o gabarito por assert.
from pyspark.sql import functions as F

etapas = {1: vendas_etapa1, 2: vendas_etapa2, 3: vendas_etapa3}

# Gabarito da seção "Janelas temporais": (início, fim) -> (receita, quantidade de vendas).
gabarito = {
    1: {
        ("2022-10-31 19:58", "2022-10-31 20:03"): (60.00, 1),
        ("2022-10-31 20:00", "2022-10-31 20:05"): (135.00, 3),
        ("2022-10-31 20:02", "2022-10-31 20:07"): (75.00, 2),
        ("2022-10-31 20:04", "2022-10-31 20:09"): (30.00, 1),
    },
    2: {
        ("2022-10-31 19:58", "2022-10-31 20:03"): (60.00, 1),
        ("2022-10-31 20:00", "2022-10-31 20:05"): (135.00, 3),
        ("2022-10-31 20:02", "2022-10-31 20:07"): (165.00, 3),
        ("2022-10-31 20:04", "2022-10-31 20:09"): (135.00, 3),
        ("2022-10-31 20:06", "2022-10-31 20:11"): (105.00, 2),
    },
    3: {
        ("2022-10-31 19:58", "2022-10-31 20:03"): (60.00, 1),
        ("2022-10-31 20:00", "2022-10-31 20:05"): (135.00, 3),
        ("2022-10-31 20:02", "2022-10-31 20:07"): (165.00, 3),
        ("2022-10-31 20:04", "2022-10-31 20:09"): (195.00, 4),
        ("2022-10-31 20:06", "2022-10-31 20:11"): (240.00, 5),
        ("2022-10-31 20:08", "2022-10-31 20:13"): (135.00, 3),
        ("2022-10-31 20:10", "2022-10-31 20:15"): (30.00, 1),
    },
}


# Uma única função faz a agregação, na versão estática e no streaming:
# se a estática confere com o gabarito, o streaming roda exatamente a mesma conta.
def receita_por_janela(df):
    # Mantive 5/2 porque são os parâmetros do Baiano e da aula.
    # Nesse formato, cada venda participa de 2 ou 3 janelas.
    return (
        df.groupBy(window("timestamp", "5 minutes", "2 minutes"))
        .agg(soma("valor").alias("receita"), F.count("*").alias("qtd_vendas"))
    )


# Datas formatadas dentro do Spark e com o dia: não dependem do fuso do collect()
# e janelas de dias diferentes não se misturam.
FORMATO = "yyyy-MM-dd HH:mm"

def formatar(df):
    return (
        df.select(
            F.date_format("window.start", FORMATO).alias("inicio"),
            F.date_format("window.end", FORMATO).alias("fim"),
            F.round("receita", 2).cast("double").alias("receita"),
            "qtd_vendas",
        )
        .orderBy("inicio")
    )


def para_dict(df):
    return {(r.inicio, r.fim): (r.receita, r.qtd_vendas) for r in formatar(df).collect()}

In [6]:
for n, vendas_da_etapa in etapas.items():
    acumulado = [v for e in range(1, n + 1) for v in etapas[e]]
    vendas = (
        spark.createDataFrame(acumulado, "timestamp string, filme string, valor int")
        .withColumn("timestamp", to_timestamp("timestamp", "yyyy-MM-dd HH:mm:ss"))
    )
    resultado = receita_por_janela(vendas)

    print(f"ETAPA {n}: {len(acumulado)} vendas acumuladas")
    formatar(resultado).show(truncate=False)

    assert para_dict(resultado) == gabarito[n], f"Etapa {n} diferente do gabarito"
    print(f"Etapa {n} confere com o gabarito\n")

ETAPA 1: 3 vendas acumuladas


+----------------+----------------+-------+----------+
|inicio          |fim             |receita|qtd_vendas|
+----------------+----------------+-------+----------+
|2022-10-31 19:58|2022-10-31 20:03|60.0   |1         |
|2022-10-31 20:00|2022-10-31 20:05|135.0  |3         |
|2022-10-31 20:02|2022-10-31 20:07|75.0   |2         |
|2022-10-31 20:04|2022-10-31 20:09|30.0   |1         |
+----------------+----------------+-------+----------+



Etapa 1 confere com o gabarito

ETAPA 2: 5 vendas acumuladas


+----------------+----------------+-------+----------+
|inicio          |fim             |receita|qtd_vendas|
+----------------+----------------+-------+----------+
|2022-10-31 19:58|2022-10-31 20:03|60.0   |1         |
|2022-10-31 20:00|2022-10-31 20:05|135.0  |3         |
|2022-10-31 20:02|2022-10-31 20:07|165.0  |3         |
|2022-10-31 20:04|2022-10-31 20:09|135.0  |3         |
|2022-10-31 20:06|2022-10-31 20:11|105.0  |2         |
+----------------+----------------+-------+----------+



Etapa 2 confere com o gabarito

ETAPA 3: 8 vendas acumuladas


+----------------+----------------+-------+----------+
|inicio          |fim             |receita|qtd_vendas|
+----------------+----------------+-------+----------+
|2022-10-31 19:58|2022-10-31 20:03|60.0   |1         |
|2022-10-31 20:00|2022-10-31 20:05|135.0  |3         |
|2022-10-31 20:02|2022-10-31 20:07|165.0  |3         |
|2022-10-31 20:04|2022-10-31 20:09|195.0  |4         |
|2022-10-31 20:06|2022-10-31 20:11|240.0  |5         |
|2022-10-31 20:08|2022-10-31 20:13|135.0  |3         |
|2022-10-31 20:10|2022-10-31 20:15|30.0   |1         |
+----------------+----------------+-------+----------+



Etapa 3 confere com o gabarito



**Leitura:** as três etapas da versão estática conferem com o gabarito calculado à mão, inclusive a quantidade de vendas em cada janela. Com isso, a janela e a soma estão validadas antes do streaming. Se a próxima parte der diferente, o problema está na chegada dos dados, e não na conta.

# Structured Streaming

## Fonte e processamento

*Os registros são simulados: não existe um sistema de vendas real por trás.*

**Fonte.** As 8 vendas da seção anterior ficam em três listas Python, uma por etapa. Cada venda tem `timestamp`, `filme` e `valor`. Os horários são lidos no fuso `America/Sao_Paulo`, definido na sessão do Spark.

**Como os registros chegam.** Cada etapa vira um arquivo JSON, com uma venda por linha e um nome único. O arquivo é escrito primeiro em uma pasta de publicação e, só depois de fechado, é movido para a pasta de entrada. Assim o Spark nunca lê um arquivo pela metade. Para o Spark, cada arquivo novo na pasta de entrada é um lote de vendas que acabou de chegar.

**Como são processados.** O `readStream` monitora a pasta de entrada com um schema fixo e trata a pasta como uma tabela que só cresce. A consulta soma `valor` por janela deslizante e grava o resultado em uma tabela em memória, no modo `complete`: a cada etapa, a tabela inteira é atualizada. O checkpoint guarda o que já foi lido, para cada venda ser contada uma vez só. Depois de publicar uma etapa, `processAllAvailable()` espera o Spark terminar de processar o que chegou e só então a tabela é consultada.

In [7]:
import json
import tempfile
from pathlib import Path
from uuid import uuid4
from pyspark.sql.types import StructType, StructField, TimestampType, IntegerType, StringType

# Adicionei esta limpeza para, se o notebook for reexecutado no mesmo kernel,
# não ficarem duas consultas ativas nem pastas de uma execução anterior.
if "query" in globals() and query.isActive:
    query.stop()
if "nome_tabela" in globals():
    spark.catalog.dropTempView(nome_tabela)
if "area_temporaria" in globals():
    area_temporaria.cleanup()

area_temporaria = tempfile.TemporaryDirectory(prefix="vendas_janelas_")
pasta_base = Path(area_temporaria.name)
pasta_entrada = pasta_base / "entrada"
pasta_publicacao = pasta_base / "publicacao"
pasta_checkpoint = pasta_base / "checkpoint"
pasta_entrada.mkdir()
pasta_publicacao.mkdir()

schema = StructType([
    StructField("timestamp", TimestampType(), True),
    StructField("filme", StringType(), True),
    # Mantive IntegerType porque todos os valores simulados são quantias inteiras.
    StructField("valor", IntegerType(), True),
])
nome_tabela = "receita_janelas_" + uuid4().hex

print("Arquivos na entrada:", list(pasta_entrada.iterdir()))

Arquivos na entrada: []


## Iniciando consulta

In [8]:
vendas_stream = (
    spark.readStream
    .schema(schema)
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
    .json(str(pasta_entrada))
)

# Troquei o groupBy que estava aqui pela receita_por_janela, a mesma da versão estática,
# para o streaming fazer exatamente a conta que já foi validada com o gabarito.
receita_stream = receita_por_janela(vendas_stream)

query = (
    receita_stream.writeStream
    .format("memory")
    .queryName(nome_tabela)
    .outputMode("complete")
    .option("checkpointLocation", str(pasta_checkpoint))
    .start()
)
print("Consulta ativa:", query.isActive)

Consulta ativa: True


In [9]:
def adicionar_vendas(novas_vendas):
    nome_arquivo = "vendas_" + uuid4().hex + ".json"
    arquivo = pasta_publicacao / nome_arquivo
    with arquivo.open("w", encoding="utf-8") as arquivo_json:
        for horario, filme, valor in novas_vendas:
            venda = {"timestamp": horario, "filme": filme, "valor": valor}
            arquivo_json.write(json.dumps(venda) + "\n")

    # Publica somente depois que o arquivo foi escrito e fechado.
    arquivo.replace(pasta_entrada / nome_arquivo)
    query.processAllAvailable()

    # Mudei para formatar(), que mostra também a quantidade de vendas
    # e as datas no mesmo formato do gabarito.
    formatar(spark.table(nome_tabela)).show(truncate=False)

In [10]:
# As células das etapas chamavam adicionar_vendas direto. Mudei para rodar_etapa,
# que publica com a mesma função, confere com o gabarito e mostra o que mudou.
historico = {}

def rodar_etapa(n):
    adicionar_vendas(etapas[n])
    atual = para_dict(spark.table(nome_tabela))
    assert atual == gabarito[n], f"Streaming da etapa {n} diferente do gabarito"

    anterior = historico.get(n - 1, {})
    novas = [janela for janela in atual if janela not in anterior]
    alteradas = [janela for janela in anterior if atual[janela] != anterior[janela]]
    historico[n] = atual

    hora = lambda janela: f"{janela[0][-5:]}–{janela[1][-5:]}"
    print(f"Etapa {n}: {len(etapas[n])} vendas chegaram; confere com o gabarito")
    print("Janelas novas:", ", ".join(hora(j) for j in novas) or "nenhuma")
    for janela in alteradas:
        print(f"Janela alterada: {hora(janela)}  R$ {anterior[janela][0]:.0f} -> R$ {atual[janela][0]:.0f}")

## Comprovando que a consulta ficou igual ao gabarito

In [11]:
rodar_etapa(1)

+----------------+----------------+-------+----------+
|inicio          |fim             |receita|qtd_vendas|
+----------------+----------------+-------+----------+
|2022-10-31 19:58|2022-10-31 20:03|60.0   |1         |
|2022-10-31 20:00|2022-10-31 20:05|135.0  |3         |
|2022-10-31 20:02|2022-10-31 20:07|75.0   |2         |
|2022-10-31 20:04|2022-10-31 20:09|30.0   |1         |
+----------------+----------------+-------+----------+



Etapa 1: 3 vendas chegaram; confere com o gabarito
Janelas novas: 19:58–20:03, 20:00–20:05, 20:02–20:07, 20:04–20:09


**Leitura:** chegaram as 3 primeiras vendas e a tabela, que estava vazia, ganhou 4 janelas. A maior é 20:00–20:05, com R$ 135, porque é a única que cobre as três vendas. A venda das 20:03 não entra em 19:58–20:03, porque o fim da janela é excluído.

In [12]:
rodar_etapa(2)

+----------------+----------------+-------+----------+
|inicio          |fim             |receita|qtd_vendas|
+----------------+----------------+-------+----------+
|2022-10-31 19:58|2022-10-31 20:03|60.0   |1         |
|2022-10-31 20:00|2022-10-31 20:05|135.0  |3         |
|2022-10-31 20:02|2022-10-31 20:07|165.0  |3         |
|2022-10-31 20:04|2022-10-31 20:09|135.0  |3         |
|2022-10-31 20:06|2022-10-31 20:11|105.0  |2         |
+----------------+----------------+-------+----------+

Etapa 2: 2 vendas chegaram; confere com o gabarito
Janelas novas: 20:06–20:11
Janela alterada: 20:02–20:07  R$ 75 -> R$ 165
Janela alterada: 20:04–20:09  R$ 30 -> R$ 135


**Leitura:** chegaram 2 vendas (20:06 e 20:07). Além de abrir a janela nova 20:06–20:11, elas aumentaram duas janelas que já existiam: 20:02–20:07 foi de R$ 75 para R$ 165 e 20:04–20:09 foi de R$ 30 para R$ 135. O streaming não recalcula tudo do zero: atualiza o estado das janelas que a venda nova toca.

In [13]:
rodar_etapa(3)

+----------------+----------------+-------+----------+
|inicio          |fim             |receita|qtd_vendas|
+----------------+----------------+-------+----------+
|2022-10-31 19:58|2022-10-31 20:03|60.0   |1         |
|2022-10-31 20:00|2022-10-31 20:05|135.0  |3         |
|2022-10-31 20:02|2022-10-31 20:07|165.0  |3         |
|2022-10-31 20:04|2022-10-31 20:09|195.0  |4         |
|2022-10-31 20:06|2022-10-31 20:11|240.0  |5         |
|2022-10-31 20:08|2022-10-31 20:13|135.0  |3         |
|2022-10-31 20:10|2022-10-31 20:15|30.0   |1         |
+----------------+----------------+-------+----------+

Etapa 3: 3 vendas chegaram; confere com o gabarito
Janelas novas: 20:08–20:13, 20:10–20:15
Janela alterada: 20:04–20:09  R$ 135 -> R$ 195
Janela alterada: 20:06–20:11  R$ 105 -> R$ 240


**Leitura:** chegaram as 3 últimas vendas. Elas abriram 20:08–20:13 e 20:10–20:15 e aumentaram 20:04–20:09 (R$ 135 → 195) e 20:06–20:11 (R$ 105 → 240). As três primeiras janelas não mudaram nesta etapa, porque nenhuma venda nova caiu no horário delas.

## Pico da noite

A agregação soma o cinema inteiro e descarta o filme. Para saber qual filme puxou a janela de maior receita, as vendas que chegaram à pasta de entrada são lidas de novo em modo batch, e só as que caem dentro dessa janela entram na conta.

In [14]:
pico = formatar(spark.table(nome_tabela)).orderBy(F.desc("receita"), "inicio").first()

vendas_recebidas = (
    spark.read.schema(schema)
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
    .json(str(pasta_entrada))
)
receita_real = vendas_recebidas.agg(soma("valor")).first()[0]
soma_janelas = formatar(spark.table(nome_tabela)).agg(soma("receita")).first()[0]

print(f"Janela de maior receita: {pico.inicio[-5:]}–{pico.fim[-5:]}, R$ {pico.receita:.0f} em {pico.qtd_vendas} vendas")
print(f"Receita real da noite: R$ {receita_real}  |  soma de todas as janelas: R$ {soma_janelas:.0f}")

# Janela é [início, fim): a venda no horário exato do fim fica de fora.
(
    vendas_recebidas
    .filter((F.col("timestamp") >= F.to_timestamp(F.lit(pico.inicio), FORMATO))
            & (F.col("timestamp") < F.to_timestamp(F.lit(pico.fim), FORMATO)))
    .groupBy("filme").agg(soma("valor").alias("receita_no_pico"))
    .orderBy(F.desc("receita_no_pico"))
    .show(truncate=False)
)

Janela de maior receita: 20:06–20:11, R$ 240 em 5 vendas
Receita real da noite: R$ 375  |  soma de todas as janelas: R$ 960


+--------------+---------------+
|filme         |receita_no_pico|
+--------------+---------------+
|Barbarian     |90             |
|Halloween Ends|60             |
|Terrifier 2   |45             |
|Smile         |45             |
+--------------+---------------+



**Leitura:** o pico é 20:06–20:11, com R$ 240 em 5 vendas, e *Barbarian* puxou essa janela com R$ 90. A soma de todas as janelas dá R$ 960, mas o cinema vendeu R$ 375 de verdade, porque cada venda é contada em 2 ou 3 janelas.

## Resultados e interpretação

*Os registros analisados são simulados: horários e valores foram inventados para a demonstração, mas os filmes existem na base do RA1.*

**O que cada etapa mostrou.** A consulta de streaming ficou ativa o tempo todo, e cada etapa só publicou um arquivo novo na pasta de entrada. A tabela de resultado mudou a cada chegada:

| Etapa | Vendas que chegaram | Janelas novas | Janelas que mudaram |
|---|---|---|---|
| 1 | 3 (20:01 a 20:04) | 4 | nenhuma (a tabela estava vazia) |
| 2 | 2 (20:06 e 20:07) | 20:06–20:11 | 20:02–20:07 (75 → 165) e 20:04–20:09 (30 → 135) |
| 3 | 3 (20:08 a 20:10) | 20:08–20:13 e 20:10–20:15 | 20:04–20:09 (135 → 195) e 20:06–20:11 (105 → 240) |

Nas três etapas o resultado do streaming foi igual ao gabarito feito à mão e à versão estática, conferido por `assert`.

**Por que uma venda aparece em mais de uma janela.** As janelas duram 5 minutos, mas uma nova começa a cada 2 minutos, então elas se sobrepõem. Cada janela vale de [início, fim): o início entra e o fim não. Uma venda entra em toda janela que cobre o horário dela, que aqui são 2 ou 3 janelas. A venda das 20:04 está em 20:00–20:05, 20:02–20:07 e 20:04–20:09. A das 20:03 está só em 20:00–20:05 e 20:02–20:07, porque em 19:58–20:03 ela cairia no fim, que é excluído. Por isso a soma de todas as janelas (R$ 960) é maior que a receita real da noite (R$ 375). Cada janela responde "quanto entrou nos últimos 5 minutos", e não é para somar as janelas entre si.

**Conclusão para o cinema.** O intervalo de maior arrecadação foi 20:06–20:11, com R$ 240, puxado por *Barbarian* (R$ 90). Neste exemplo, as janelas antigas deixam de mudar porque as vendas simuladas chegam em ordem cronológica. Sem watermark, uma venda atrasada ainda poderia atualizá-las.

**Limitação.** O resultado fica em uma tabela em memória (`format("memory")`), que serve para demonstração com pouco volume. Ela não guarda nada depois que a sessão termina e não é o que se usaria em produção. A consulta também não usa watermark: o conjunto é pequeno, e o modo `complete` precisa manter a tabela inteira para mostrá-la a cada etapa, então não haveria estado para descartar.

## Encerramento

Para a consulta e apaga a área temporária com as pastas de entrada, publicação e checkpoint.

In [15]:
query.stop()
spark.catalog.dropTempView(nome_tabela)
area_temporaria.cleanup()

print("Consulta ativa:", query.isActive)
print("Pastas temporárias existem:", pasta_base.exists())

Consulta ativa: False
Pastas temporárias existem: False
